# Multi-modal Model V2.5 - Fast GRU & Y-Flip Augmentation

이 노트북은 `MultiModal_v2_4`의 속도 이슈를 해결하고, 데이터 증강을 통해 성능을 극대화하는 버전입니다.

## 주요 변경 사항 (V2.5)

1.  **GRU Architecture (Speed)**:
    - LSTM 대신 구조가 단순한 **GRU (Gated Recurrent Unit)**를 사용합니다.
    - 파라미터 수가 적어 학습 속도가 빠르고, 적은 데이터셋에서 과적합 위험이 적습니다.
2.  **Random Y-Flip Augmentation (Robustness)**:
    - 축구장은 Y축(가로 중심선) 기준으로 대칭입니다.
    - 학습 시 **50% 확률로 위아래를 뒤집어(Flip)** 데이터를 2배로 늘리는 효과를 냅니다.
3.  **Visualization Revert (Efficiency)**:
    - v2.4의 속도 저하 원인이었던 Thick Trajectory를 제거하고, 다시 **1px 선 그리기**로 복귀하여 빠른 데이터 로딩 속도를 확보합니다.

## 모델 개요
- **Feature**: v2.2와 동일 (안정적 베이스라인)
- **Model**: CNN + **Bi-GRU** + Fusion
- **Augmentation**: Random Y-Flip (Train Only)

In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from sklearn.model_selection import GroupKFold
import cv2  # OpenCV Import
from tqdm import tqdm
import joblib

# [v7 Fix] Use Fixed Preprocessor (No Rounding)
from src.preprocessing_multimodal_fixed import FootballPreprocessorMultimodal

# Fold-specific Seeds for Diversity
FOLD_SEEDS = {
    1: 42,
    2: 43,
    3: 44,
    4: 45,
    5: 46,
    6: 47,
    7: 48,
    8: 49,
    9: 50,
    10: 51,
}

# Set Seed
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Device: cuda


## 1. 데이터 로드 및 전처리 (Preprocessing V2.5)

## 2. 이미지 생성 및 증강 (Random Y-Flip)

In [2]:
# Load Raw Training Data
train_df = pd.read_csv("train.csv")
print(f"Loaded {len(train_df)} rows")

Loaded 356721 rows


In [3]:
class MultiModalDataset(Dataset):
    def __init__(self, episodes, img_size=(68, 105), augment=False, cache_images=True):
        self.episodes = episodes
        self.H, self.W = img_size
        self.augment = augment
        self.cache_images = cache_images
        self.img_cache = None
        
        # [v7 Fix] Image Caching (Speed Optimization)
        if self.cache_images:
            print(f"Pre-rendering {len(episodes)} images (Cache Enabled)...")
            # Generate Base Images (No Augmentation applied yet)
            self.img_cache = [self._generate_image(ep['cont']) for ep in tqdm(episodes, desc="Caching Images")]

    def __len__(self):
        return len(self.episodes)

    def _generate_image(self, cont_data):
        # [v6 Optimization] Fast Generation with OpenCV
        img_np = np.zeros((2, self.H, self.W), dtype=np.float32)
        if len(cont_data) == 0:
             return torch.tensor(img_np, dtype=torch.float32)

        seq_len = cont_data.shape[0]
        
        xs = cont_data[:, 0]
        ys = cont_data[:, 1]
        
        # 좌표 변환 (0~1 -> Grid)
        # Note: Input is float (fixed preprocessor), map to int index
        x_idxs = (xs * (self.W - 1)).round().astype(int).clip(0, self.W - 1)
        y_idxs = (ys * (self.H - 1)).round().astype(int).clip(0, self.H - 1)
        
        for t in range(seq_len):
            px, py = x_idxs[t], y_idxs[t]
            
            # Channel 0: Dots
            img_np[0, py, px] = 1.0
            
            decay_val = (t + 1) / seq_len
            
            # Channel 1: Line
            if t > 0:
                prev_px, prev_py = x_idxs[t-1], y_idxs[t-1]
                cv2.line(img_np[1], (prev_px, prev_py), (px, py), color=decay_val, thickness=1)
            else:
                img_np[1, py, px] = decay_val
        return torch.tensor(img_np, dtype=torch.float32)

    def __getitem__(self, idx):
        data = self.episodes[idx]
        
        cont = data['cont'].copy()
        target = data['target'].copy()
        
        # Augmentation Logic (Y-Flip)
        if self.augment and random.random() < 0.5:
            # 1. Flip Continuous Features
            cont[:, 1] = 1.0 - cont[:, 1] # start_y
            cont[:, 3] = 1.0 - cont[:, 3] # end_y_prev
            cont[:, 5] = -cont[:, 5]      # dy_prev (Vector Flip)
            target[1] = 1.0 - target[1]   # Target Flip
            
            # 2. Get Image
            if self.cache_images:
                # Retrieve Base Image from Cache
                base_img = self.img_cache[idx]
                # Apply Flip (Y-Flip = Flip H dimension = dim 1)
                img = torch.flip(base_img, [1])
            else:
                img = self._generate_image(cont) # Generate from flipped coords
        else:
            # No Augmentation
            if self.cache_images:
                img = self.img_cache[idx]
            else:
                img = self._generate_image(cont)
        
        cont_tensor = torch.tensor(cont, dtype=torch.float32)
        cat_tensor = torch.tensor(data['cat'], dtype=torch.long)
        target_tensor = torch.tensor(target, dtype=torch.float32)
        
        # Deep Supervision Targets
        seq_len = cont.shape[0]
        aux_target = np.zeros((seq_len, 2), dtype=np.float32)
        if seq_len > 1:
            aux_target[:-1] = cont[1:, 0:2]
        aux_target[-1] = target # Last step target is GT
        aux_target_tensor = torch.tensor(aux_target, dtype=torch.float32)
        
        return img, cont_tensor, cat_tensor, target_tensor, aux_target_tensor

## 3. 모델 아키텍처 (GRU 적용)

In [4]:
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        x_out = self.conv(x_cat)
        return self.sigmoid(x_out)

class ImprovedCNN(nn.Module):
    def __init__(self):
        super(ImprovedCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
        )
        self.sa = SpatialAttention()
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, 128)
        
    def forward(self, x):
        x = self.features(x)
        sa_map = self.sa(x)
        x = x * sa_map
        x = self.pool(x).flatten(1)
        x = F.relu(self.fc(x))
        return x

class LSTMAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(LSTMAttention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)
    def forward(self, rnn_output):
        attn_weights = torch.softmax(self.attention(rnn_output), dim=1)
        return torch.sum(attn_weights * rnn_output, dim=1)

class MultiModalNetV6(nn.Module):
    def __init__(self, input_dim_cont, num_types, num_results, gru_hidden=96):
        super(MultiModalNetV6, self).__init__()
        self.cnn = ImprovedCNN()
        self.type_emb = nn.Embedding(num_types, 8)
        self.result_emb = nn.Embedding(num_results, 8)
        
        total_input_dim = input_dim_cont + 8 + 8
        self.gru_hidden = gru_hidden
        
        # [변경] Split GRU (Anti-Leakage)
        # Forward와 Backward를 물리적으로 분리하여, Forward GRU에 미래 정보가 유입되는 것을 원천 차단
        self.gru_fwd = nn.GRU(
            input_size=total_input_dim,
            hidden_size=gru_hidden,
            num_layers=2, # Multi-layer OK (Forward끼리만 연결됨)
            batch_first=True,
            bidirectional=False, # 단방향
            dropout=0.1
        )
        
        self.gru_bwd = nn.GRU(
            input_size=total_input_dim,
            hidden_size=gru_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=False, # 단방향
            dropout=0.1
        )
        
        self.gru_attn = LSTMAttention(gru_hidden * 2)
        self.gru_fc = nn.Linear(gru_hidden * 2, 128)
        
        # [New] Deep Supervision Head
        # Use ONLY Forward Hidden State for Aux Output
        self.aux_fc = nn.Linear(gru_hidden, 2) 
        
        self.fusion_fc = nn.Sequential(
            nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, img, cont, cat, lengths):
        # 1. Feature Extraction
        img_feat = self.cnn(img)
        emb_type = self.type_emb(cat[:, :, 0])
        emb_result = self.result_emb(cat[:, :, 1])
        x_seq = torch.cat([cont, emb_type, emb_result], dim=2)
        
        # 2. Forward GRU
        packed_fwd = pack_padded_sequence(x_seq, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out_fwd_packed, _ = self.gru_fwd(packed_fwd)
        out_fwd, _ = pad_packed_sequence(out_fwd_packed, batch_first=True)
        
        # 3. Backward GRU (Manual Flip)
        # Reverse sequence for backward pass
        # Note: Padding should remain at the end, so we need careful reversal?
        # pack_padded_sequence handles lengths correctly, verifying...
        # For simplicity and correctness with variable lengths:
        # We can just reverse the input tensor in time dim, pack, run GRU, unpack, reverse back.
        # But we need to handle padding.
        
        # Optimized Backward Strategy:
        # Since we use pack_padded_sequence with enforce_sorted=False, 
        # we can just flip the non-padded parts? Or just flip the whole tensor and let pack handle it?
        # Actually, PyTorch's pad_packed_sequence output is padded with zeros.
        # It's safer to rely on PyTorch's bidirectional implementation but we can't because of the leakage.
        # So manual flip is needed.
        
        # Manual Flip Logic:
        x_seq_bwd = x_seq.clone()
        for i, length in enumerate(lengths):
            x_seq_bwd[i, :length] = x_seq[i, :length].flip(0)
            
        packed_bwd = pack_padded_sequence(x_seq_bwd, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out_bwd_packed, _ = self.gru_bwd(packed_bwd)
        out_bwd, _ = pad_packed_sequence(out_bwd_packed, batch_first=True)
        
        # Reverse output back to original order
        for i, length in enumerate(lengths):
            out_bwd[i, :length] = out_bwd[i, :length].flip(0)
            
        # 4. Concatenate: [Forward, Backward]
        # out_fwd: [B, L, H], out_bwd: [B, L, H]
        gru_out_combined = torch.cat([out_fwd, out_bwd], dim=2) # [B, L, H*2]
        
        # 5. Main Output (Legacy)
        gru_ctx = self.gru_attn(gru_out_combined)
        seq_feat = F.relu(self.gru_fc(gru_ctx))
        concat_feat = torch.cat([img_feat, seq_feat], dim=1)
        final_out = self.fusion_fc(concat_feat)
        
        # 6. Aux Output (Deep Supervision)
        # Use ONLY out_fwd (Pure Past->Future info)
        aux_out = self.aux_fc(out_fwd) # [B, L, 2]
        
        return final_out, aux_out

class EuclideanLoss(nn.Module):
    def __init__(self):
        super(EuclideanLoss, self).__init__()
    def forward(self, pred, target):
        pred_real = pred * torch.tensor([105.0, 68.0], device=pred.device)
        target_real = target * torch.tensor([105.0, 68.0], device=target.device)
        return torch.mean(torch.sqrt(torch.sum((pred_real - target_real)**2, dim=1) + 1e-6))
        
class MaskedSeqEuclideanLoss(nn.Module):
    def __init__(self):
        super(MaskedSeqEuclideanLoss, self).__init__()
        
    def forward(self, pred, target, lengths):
        # pred: [B, L, 2]
        # target: [B, L, 2]
        # lengths: [B]
        
        mask = torch.arange(pred.size(1), device=pred.device)[None, :] < lengths[:, None]
        mask = mask.unsqueeze(-1) # [B, L, 1]
        
        pred_real = pred * torch.tensor([105.0, 68.0], device=pred.device)
        target_real = target * torch.tensor([105.0, 68.0], device=target.device)
        
        diff = pred_real - target_real
        dist = torch.sqrt(torch.sum(diff**2, dim=2) + 1e-6) # [B, L]
        
        # Masking
        dist = dist * mask.squeeze(-1)
        
        return dist.sum() / mask.sum()

## 4. 학습

In [5]:
def multimodal_collate_fn(batch):
    imgs, conts, cats, targets, aux_targets = zip(*batch)
    imgs_batched = torch.stack(imgs, dim=0)
    lengths = torch.tensor([len(c) for c in conts], dtype=torch.long)
    conts_padded = pad_sequence(conts, batch_first=True)
    cats_padded = pad_sequence(cats, batch_first=True)
    targets = torch.stack(targets, dim=0)
    aux_targets_padded = pad_sequence(aux_targets, batch_first=True)
    return imgs_batched, conts_padded, cats_padded, lengths, targets, aux_targets_padded

In [6]:
# [v7 Fix] Leakage-Free Training Function
# Now receives Raw DataFrame instead of transformed episodes
def train_multimodal_v7(train_df, n_splits=10, epochs=100, batch_size=64, lr=0.001):
    gkf = GroupKFold(n_splits=n_splits)
    groups = train_df['game_id'] 
    
    fold_scores = []

    # Get dimensions from first fold (will be same for all folds)
    input_dim_cont = None
    num_types = None
    num_results = None
    
    # K-Fold Loop
    for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, groups=groups)):
        
        # Set fold-specific seed for diversity
        fold_seed = FOLD_SEEDS[fold + 1]
        seed_everything(fold_seed)
        print(f"[Fold {fold+1}] Seed fixed to {fold_seed}")
        print(f"\n=== Fold {fold+1}/{n_splits} ===")
        
        # 1. Split Raw Data
        train_sub_df = train_df.iloc[train_idx].copy()
        val_sub_df = train_df.iloc[val_idx].copy()
        
        # 2. Fit Preprocessor (Inside Loop -> No Leakage)
        preprocessor = FootballPreprocessorMultimodal()
        preprocessor.fit(train_sub_df)

        # Get dimensions (only once, on first fold)
        if input_dim_cont is None:
            input_dim_cont = preprocessor.get_input_dim()
            num_types, num_results = preprocessor.get_num_classes()
            print(f"Model Dimensions: input={input_dim_cont}, types={num_types}, results={num_results}")
        
        # 3. Transform Data
        # 3. Transform Data
        train_episodes = preprocessor.transform(train_sub_df, is_train=True)
        val_episodes = preprocessor.transform(val_sub_df, is_train=True) # Val also needs targets
        
        input_dim_cont = preprocessor.get_input_dim()
        num_types, num_results = preprocessor.get_num_classes()
        
        # 4. Dataset & Loader (With Caching)
        # Train: Cache Enabled, Augment Enabled
        train_dataset = MultiModalDataset(train_episodes, augment=True, cache_images=True)
        # Val: Cache Enabled, No Augment
        val_dataset = MultiModalDataset(val_episodes, augment=False, cache_images=True)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=multimodal_collate_fn, drop_last=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=multimodal_collate_fn)
        
        # Model (Split-GRU v6 Architecture)
        model = MultiModalNetV6(input_dim_cont, num_types, num_results).to(DEVICE)
        
        criterion_main = EuclideanLoss()
        criterion_aux = MaskedSeqEuclideanLoss() 
        
        optimizer = optim.Adam(model.parameters(), lr=lr) 
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
        
        best_dist = float('inf')
        best_state = None
        patience_counter = 0
        patience_limit = 15
        
        for epoch in range(epochs):
            model.train()
            train_loss = 0
            
            # Tqdm for progress tracking (optional, maybe noisy for user logs)
            # for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            for batch in train_loader:
                imgs, cont, cat, lengths, target, aux_target = batch
                imgs, cont, cat, lengths, target, aux_target = imgs.to(DEVICE), cont.to(DEVICE), cat.to(DEVICE), lengths.to(DEVICE), target.to(DEVICE), aux_target.to(DEVICE)
                
                optimizer.zero_grad()
                pred, aux_pred = model(imgs, cont, cat, lengths)
                
                loss_m = criterion_main(pred, target)
                loss_a = criterion_aux(aux_pred, aux_target, lengths)
                
                loss = loss_m + 0.5 * loss_a
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            
            train_loss /= len(train_loader)
            
            model.eval()
            val_dists = []
            with torch.no_grad():
                for batch in val_loader:
                    imgs, cont, cat, lengths, target, aux_target = batch
                    imgs, cont, cat, lengths, target, aux_target = imgs.to(DEVICE), cont.to(DEVICE), cat.to(DEVICE), lengths.to(DEVICE), target.to(DEVICE), aux_target.to(DEVICE)
                    
                    pred, _ = model(imgs, cont, cat, lengths)
                    
                    pred_real = pred.cpu().numpy() * np.array([105.0, 68.0])
                    target_real = target.cpu().numpy() * np.array([105.0, 68.0])
                    val_dists.extend(np.sqrt(np.sum((pred_real - target_real)**2, axis=1)))
            
            mean_dist = np.mean(val_dists)
            scheduler.step(mean_dist)
            print(f"Epoch {epoch+1}: Train Loss {train_loss:.4f}, Val Dist {mean_dist:.4f}")
            
            if mean_dist < best_dist:
                best_dist = mean_dist
                best_state = model.state_dict()
                # Save with v7 suffix
                torch.save(best_state, f"models/multimodal_v7_h96_{fold+1}.pth")
                joblib.dump(preprocessor, f"models/preprocessor_v7_h96_{fold+1}.pkl")
                print(f"  -> Saved Best Model (Dist: {best_dist:.4f})")
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience_limit:
                    print("Early Stopping")
                    break
        
        print(f"Fold {fold+1} Finished. Best Valid Dist: {best_dist:.4f}")
        fold_scores.append(best_dist)
        
    print(f"Average Score: {np.mean(fold_scores):.4f}")

In [7]:
# [v7 Execution]
train_multimodal_v7(train_df, n_splits=10, epochs=100, batch_size=32, lr=0.001)

[Fold 1] Seed fixed to 42

=== Fold 1/10 ===
Model Dimensions: input=10, types=27, results=21
Pre-rendering 13847 images (Cache Enabled)...


Caching Images: 100%|██████████| 13847/13847 [00:01<00:00, 12328.81it/s]


Pre-rendering 1581 images (Cache Enabled)...


Caching Images: 100%|██████████| 1581/1581 [00:00<00:00, 12512.21it/s]


Epoch 1: Train Loss 61.7265, Val Dist 22.2160
  -> Saved Best Model (Dist: 22.2160)
Epoch 2: Train Loss 49.2726, Val Dist 20.5656
  -> Saved Best Model (Dist: 20.5656)
Epoch 3: Train Loss 46.7672, Val Dist 17.7305
  -> Saved Best Model (Dist: 17.7305)
Epoch 4: Train Loss 45.3608, Val Dist 17.2045
  -> Saved Best Model (Dist: 17.2045)
Epoch 5: Train Loss 44.4426, Val Dist 16.6557
  -> Saved Best Model (Dist: 16.6557)
Epoch 6: Train Loss 43.6645, Val Dist 15.5423
  -> Saved Best Model (Dist: 15.5423)
Epoch 7: Train Loss 42.8739, Val Dist 15.3880
  -> Saved Best Model (Dist: 15.3880)
Epoch 8: Train Loss 42.4313, Val Dist 15.2548
  -> Saved Best Model (Dist: 15.2548)
Epoch 9: Train Loss 41.9955, Val Dist 14.8284
  -> Saved Best Model (Dist: 14.8284)
Epoch 10: Train Loss 41.7008, Val Dist 14.9495
Epoch 11: Train Loss 41.3675, Val Dist 14.6827
  -> Saved Best Model (Dist: 14.6827)
Epoch 12: Train Loss 41.0648, Val Dist 14.6355
  -> Saved Best Model (Dist: 14.6355)
Epoch 13: Train Loss 40.782

...

Epoch 1: Train Loss 60.9232, Val Dist 23.3203
  -> Saved Best Model (Dist: 23.3203)
Epoch 2: Train Loss 48.1708, Val Dist 18.5484
  -> Saved Best Model (Dist: 18.5484)
Epoch 3: Train Loss 46.1194, Val Dist 16.8056
  -> Saved Best Model (Dist: 16.8056)
Epoch 4: Train Loss 44.9212, Val Dist 17.2141
Epoch 5: Train Loss 44.0632, Val Dist 16.6737
  -> Saved Best Model (Dist: 16.6737)
Epoch 6: Train Loss 43.2871, Val Dist 15.9448
  -> Saved Best Model (Dist: 15.9448)
Epoch 7: Train Loss 42.6463, Val Dist 16.0539
Epoch 8: Train Loss 42.3506, Val Dist 16.2115
Epoch 9: Train Loss 41.9578, Val Dist 15.2220
  -> Saved Best Model (Dist: 15.2220)
Epoch 10: Train Loss 41.6495, Val Dist 15.0458
  -> Saved Best Model (Dist: 15.0458)
Epoch 11: Train Loss 41.3472, Val Dist 14.9750
  -> Saved Best Model (Dist: 14.9750)
Epoch 12: Train Loss 41.0516, Val Dist 15.1179
Epoch 13: Train Loss 40.7314, Val Dist 15.0812
Epoch 14: Train Loss 40.5792, Val Dist 14.9730
  -> Saved Best Model (Dist: 14.9730)
Epoch 15:

Caching Images: 100%|██████████| 13858/13858 [00:01<00:00, 13203.17it/s]


Pre-rendering 1570 images (Cache Enabled)...


Caching Images: 100%|██████████| 1570/1570 [00:00<00:00, 13920.01it/s]


Epoch 1: Train Loss 61.4081, Val Dist 19.5568
  -> Saved Best Model (Dist: 19.5568)
Epoch 2: Train Loss 48.3333, Val Dist 17.6906
  -> Saved Best Model (Dist: 17.6906)
Epoch 3: Train Loss 46.1646, Val Dist 16.5409
  -> Saved Best Model (Dist: 16.5409)
Epoch 4: Train Loss 44.7186, Val Dist 16.4845
  -> Saved Best Model (Dist: 16.4845)
Epoch 5: Train Loss 43.9346, Val Dist 15.2780
  -> Saved Best Model (Dist: 15.2780)
Epoch 6: Train Loss 43.2475, Val Dist 15.5605
Epoch 7: Train Loss 42.7104, Val Dist 15.3338
Epoch 8: Train Loss 42.2584, Val Dist 16.0877
Epoch 9: Train Loss 41.9622, Val Dist 14.9958
  -> Saved Best Model (Dist: 14.9958)
Epoch 10: Train Loss 41.5859, Val Dist 14.8706
  -> Saved Best Model (Dist: 14.8706)
Epoch 11: Train Loss 41.4411, Val Dist 14.6179
  -> Saved Best Model (Dist: 14.6179)
Epoch 12: Train Loss 41.1461, Val Dist 15.0519
Epoch 13: Train Loss 40.8704, Val Dist 14.8316
Epoch 14: Train Loss 40.7849, Val Dist 14.5292
  -> Saved Best Model (Dist: 14.5292)
Epoch 15:

## 5. 추론 (Inference)

In [8]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import cv2
from tqdm import tqdm

os.makedirs("submission_multimodal", exist_ok=True)

# =========================
# Dataset (기존과 동일)
# =========================
class TestMultiModalDataset(Dataset):
    def __init__(self, submission_df, preprocessor, img_size=(68, 105)):
        self.submission_df = submission_df
        self.preprocessor = preprocessor
        self.H, self.W = img_size

    def __len__(self):
        return len(self.submission_df)

    def _generate_image(self, cont_data):
        img_np = np.zeros((2, self.H, self.W), dtype=np.float32)
        if len(cont_data) == 0:
            return torch.tensor(img_np, dtype=torch.float32)

        seq_len = cont_data.shape[0]
        xs = cont_data[:, 0]
        ys = cont_data[:, 1]

        x_idxs = (xs * (self.W - 1)).round().astype(int).clip(0, self.W - 1)
        y_idxs = (ys * (self.H - 1)).round().astype(int).clip(0, self.H - 1)

        for t in range(seq_len):
            px, py = x_idxs[t], y_idxs[t]
            img_np[0, py, px] = 1.0
            decay_val = (t + 1) / seq_len
            if t > 0:
                prev_px, prev_py = x_idxs[t-1], y_idxs[t-1]
                cv2.line(img_np[1], (prev_px, prev_py), (px, py), color=decay_val, thickness=1)
            else:
                img_np[1, py, px] = decay_val

        return torch.tensor(img_np, dtype=torch.float32)

    def __getitem__(self, idx):
        row = self.submission_df.iloc[idx]
        raw_path = row['path']
        path = raw_path[2:] if raw_path.startswith("./") else raw_path

        df = pd.read_csv(path)
        episodes = self.preprocessor.transform(df, is_train=False)

        if len(episodes) == 0:
            input_dim = self.preprocessor.get_input_dim()
            cont = torch.zeros((1, input_dim), dtype=torch.float32)
            cat = torch.zeros((1, 2), dtype=torch.long)
            img = torch.zeros((2, self.H, self.W), dtype=torch.float32)
        else:
            data = episodes[0]
            cont_np = data['cont']
            img = self._generate_image(cont_np)
            cont = torch.tensor(cont_np, dtype=torch.float32)
            cat = torch.tensor(data['cat'], dtype=torch.long)

        return img, cont, cat


def test_mm_collate_fn(batch):
    imgs, conts, cats = zip(*batch)
    imgs_batched = torch.stack(imgs, dim=0)
    lengths = torch.tensor([len(c) for c in conts], dtype=torch.long)
    conts_padded = pad_sequence(conts, batch_first=True)
    cats_padded = pad_sequence(cats, batch_first=True)
    return imgs_batched, conts_padded, cats_padded, lengths


# =========================
# Load Submission Meta
# =========================
submission = pd.read_csv("sample_submission.csv")
test_meta = pd.read_csv("test.csv")
submission = submission.merge(test_meta, on="game_episode", how="left")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# Fold-wise Inference
# =========================
NUM_FOLDS = 10
all_fold_preds = []

for fold in range(1, NUM_FOLDS + 1):
    print(f"\n=== Inference Fold {fold} ===")

    # 1. Load preprocessor
    preprocessor = joblib.load(f"models/preprocessor_v7_h96_{fold}.pkl")

    # 2. Dataset / Loader
    test_dataset = TestMultiModalDataset(submission, preprocessor)
    test_loader = DataLoader(
        test_dataset,
        batch_size=128,
        shuffle=False,
        collate_fn=test_mm_collate_fn
    )

    # 3. Load model
    input_dim_cont = preprocessor.get_input_dim()
    num_types, num_results = preprocessor.get_num_classes()

    model = MultiModalNetV6(input_dim_cont, num_types, num_results).to(DEVICE)
    model.load_state_dict(torch.load(f"models/multimodal_v7_h96_{fold}.pth"))
    model.eval()

    fold_preds = []

    # 4. Inference
    with torch.no_grad():
        for imgs, cont, cat, lengths in tqdm(test_loader):
            imgs, cont, cat, lengths = (
                imgs.to(DEVICE),
                cont.to(DEVICE),
                cat.to(DEVICE),
                lengths.to(DEVICE),
            )

            pred, _ = model(imgs, cont, cat, lengths)
            pred_np = pred.cpu().numpy()
            pred_np[:, 0] *= 105.0
            pred_np[:, 1] *= 68.0
            fold_preds.append(pred_np)

    fold_preds = np.vstack(fold_preds)
    all_fold_preds.append(fold_preds)

# =========================
# Ensemble (Mean)
# =========================
final_preds = np.mean(all_fold_preds, axis=0)

submission["end_x"] = final_preds[:, 0].clip(0, 105)
submission["end_y"] = final_preds[:, 1].clip(0, 68)

submission[["game_episode", "end_x", "end_y"]].to_csv(
    "submission_multimodal/multimodal_v7_h96.csv", index=False
)

print("Saved submission_multimodal/multimodal_v7_h96.csv")


C:\Users\semic\AppData\Local\Temp\ipykernel_83332\2167706117.py:116: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"models/multimodal_v7_h9


=== Inference Fold 1 ===


100%|██████████| 19/19 [01:08<00:00,  3.58s/it]
C:\Users\semic\AppData\Local\Temp\ipykernel_83332\2167706117.py:116: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load


=== Inference Fold 2 ===


100%|██████████| 19/19 [00:41<00:00,  2.21s/it]
C:\Users\semic\AppData\Local\Temp\ipykernel_83332\2167706117.py:116: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load

...


=== Inference Fold 9 ===


100%|██████████| 19/19 [00:41<00:00,  2.19s/it]
C:\Users\semic\AppData\Local\Temp\ipykernel_83332\2167706117.py:116: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load


=== Inference Fold 10 ===


100%|██████████| 19/19 [00:41<00:00,  2.19s/it]

Saved submission_multimodal/multimodal_v7_h96.csv


In [9]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import cv2
from tqdm import tqdm

os.makedirs("submission_multimodal", exist_ok=True)

# =========================
# Dataset
# =========================
class TestMultiModalDataset(Dataset):
    def __init__(self, submission_df, preprocessor, img_size=(68, 105)):
        self.submission_df = submission_df
        self.preprocessor = preprocessor
        self.H, self.W = img_size

    def __len__(self):
        return len(self.submission_df)

    def _generate_image(self, cont_data):
        img_np = np.zeros((2, self.H, self.W), dtype=np.float32)
        if len(cont_data) == 0:
            return torch.tensor(img_np, dtype=torch.float32)

        seq_len = cont_data.shape[0]
        xs = cont_data[:, 0]
        ys = cont_data[:, 1]

        x_idxs = (xs * (self.W - 1)).round().astype(int).clip(0, self.W - 1)
        y_idxs = (ys * (self.H - 1)).round().astype(int).clip(0, self.H - 1)

        for t in range(seq_len):
            px, py = x_idxs[t], y_idxs[t]
            img_np[0, py, px] = 1.0
            decay_val = (t + 1) / seq_len
            if t > 0:
                prev_px, prev_py = x_idxs[t-1], y_idxs[t-1]
                cv2.line(img_np[1], (prev_px, prev_py), (px, py), color=decay_val, thickness=1)
            else:
                img_np[1, py, px] = decay_val

        return torch.tensor(img_np, dtype=torch.float32)

    def __getitem__(self, idx):
        row = self.submission_df.iloc[idx]
        raw_path = row['path']
        path = raw_path[2:] if raw_path.startswith("./") else raw_path

        df = pd.read_csv(path)
        episodes = self.preprocessor.transform(df, is_train=False)

        if len(episodes) == 0:
            input_dim = self.preprocessor.get_input_dim()
            cont = torch.zeros((1, input_dim), dtype=torch.float32)
            cat = torch.zeros((1, 2), dtype=torch.long)
            img = torch.zeros((2, self.H, self.W), dtype=torch.float32)
        else:
            data = episodes[0]
            cont_np = data['cont']
            img = self._generate_image(cont_np)
            cont = torch.tensor(cont_np, dtype=torch.float32)
            cat = torch.tensor(data['cat'], dtype=torch.long)

        return img, cont, cat


def test_mm_collate_fn(batch):
    imgs, conts, cats = zip(*batch)
    imgs_batched = torch.stack(imgs, dim=0)
    lengths = torch.tensor([len(c) for c in conts], dtype=torch.long)
    conts_padded = pad_sequence(conts, batch_first=True)
    cats_padded = pad_sequence(cats, batch_first=True)
    return imgs_batched, conts_padded, cats_padded, lengths


# =========================
# Load Submission Meta
# =========================
submission = pd.read_csv("sample_submission.csv")
test_meta = pd.read_csv("test.csv")
submission = submission.merge(test_meta, on="game_episode", how="left")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# 10-Fold Inference with TTA
# =========================
NUM_FOLDS = 10
all_fold_preds = []

for fold in range(1, NUM_FOLDS + 1):
    print(f"\n=== Inference Fold {fold} with TTA ===")

    # 1. Load preprocessor
    preprocessor = joblib.load(f"models/preprocessor_v7_h96_{fold}.pkl")

    # 2. Dataset / Loader
    test_dataset = TestMultiModalDataset(submission, preprocessor)
    test_loader = DataLoader(
        test_dataset,
        batch_size=128,
        shuffle=False,
        collate_fn=test_mm_collate_fn
    )

    # 3. Load model
    input_dim_cont = preprocessor.get_input_dim()
    num_types, num_results = preprocessor.get_num_classes()

    model = MultiModalNetV6(input_dim_cont, num_types, num_results).to(DEVICE)
    model.load_state_dict(torch.load(f"models/multimodal_v7_h96_{fold}.pth", map_location=DEVICE))
    model.eval()

    fold_preds = []

    # 4. Inference with TTA
    with torch.no_grad():
        for imgs, cont, cat, lengths in tqdm(test_loader, desc=f"Fold {fold}"):
            imgs, cont, cat, lengths = (
                imgs.to(DEVICE),
                cont.to(DEVICE),
                cat.to(DEVICE),
                lengths.to(DEVICE),
            )

            # -------------------------
            # (1) 원본 예측
            # -------------------------
            pred1, _ = model(imgs, cont, cat, lengths)

            # -------------------------
            # (2) Y-Flip TTA 예측
            # -------------------------
            imgs_f = torch.flip(imgs, dims=[2])  # Y-axis flip

            cont_f = cont.clone()
            # Y 관련 feature flip
            cont_f[:, :, 1] = 1.0 - cont_f[:, :, 1]  # start_y
            cont_f[:, :, 3] = 1.0 - cont_f[:, :, 3]  # end_y_prev
            cont_f[:, :, 5] = -cont_f[:, :, 5]       # dy_prev

            pred2, _ = model(imgs_f, cont_f, cat, lengths)

            # 예측값 다시 flip
            pred2[:, 1] = 1.0 - pred2[:, 1]

            # -------------------------
            # (3) 평균
            # -------------------------
            pred = 0.5 * (pred1 + pred2)

            pred_np = pred.cpu().numpy()
            pred_np[:, 0] *= 105.0
            pred_np[:, 1] *= 68.0
            fold_preds.append(pred_np)

    fold_preds = np.vstack(fold_preds)
    all_fold_preds.append(fold_preds)

# =========================
# Ensemble (Mean)
# =========================
final_preds = np.mean(all_fold_preds, axis=0)

submission["end_x"] = final_preds[:, 0].clip(0, 105)
submission["end_y"] = final_preds[:, 1].clip(0, 68)

submission[["game_episode", "end_x", "end_y"]].to_csv(
    "submission_multimodal/multimodal_v7_h96_tta.csv", index=False
)

print("\n✅ Saved: submission_multimodal/multimodal_v7_h96_tta.csv")

C:\Users\semic\AppData\Local\Temp\ipykernel_83332\2450104897.py:116: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"models/multimodal_v7_h9


=== Inference Fold 1 with TTA ===


Fold 1: 100%|██████████| 19/19 [00:42<00:00,  2.22s/it]
C:\Users\semic\AppData\Local\Temp\ipykernel_83332\2450104897.py:116: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mo


=== Inference Fold 2 with TTA ===


Fold 2: 100%|██████████| 19/19 [00:43<00:00,  2.31s/it]
C:\Users\semic\AppData\Local\Temp\ipykernel_83332\2450104897.py:116: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mo

...


=== Inference Fold 9 with TTA ===


Fold 9: 100%|██████████| 19/19 [00:43<00:00,  2.30s/it]
C:\Users\semic\AppData\Local\Temp\ipykernel_83332\2450104897.py:116: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mo


=== Inference Fold 10 with TTA ===


Fold 10: 100%|██████████| 19/19 [00:43<00:00,  2.30s/it]


✅ Saved: submission_multimodal/multimodal_v7_h96_tta.csv


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import cv2
from tqdm import tqdm

os.makedirs("submission_multimodal", exist_ok=True)

# =========================
# Dataset
# =========================
class TestMultiModalDataset(Dataset):
    def __init__(self, submission_df, preprocessor, img_size=(68, 105)):
        self.submission_df = submission_df
        self.preprocessor = preprocessor
        self.H, self.W = img_size

    def __len__(self):
        return len(self.submission_df)

    def _generate_image(self, cont_data):
        img_np = np.zeros((2, self.H, self.W), dtype=np.float32)
        if len(cont_data) == 0:
            return torch.tensor(img_np, dtype=torch.float32)

        seq_len = cont_data.shape[0]
        xs = cont_data[:, 0]
        ys = cont_data[:, 1]

        x_idxs = (xs * (self.W - 1)).round().astype(int).clip(0, self.W - 1)
        y_idxs = (ys * (self.H - 1)).round().astype(int).clip(0, self.H - 1)

        for t in range(seq_len):
            px, py = x_idxs[t], y_idxs[t]
            img_np[0, py, px] = 1.0
            decay_val = (t + 1) / seq_len
            if t > 0:
                prev_px, prev_py = x_idxs[t-1], y_idxs[t-1]
                cv2.line(img_np[1], (prev_px, prev_py), (px, py), color=decay_val, thickness=1)
            else:
                img_np[1, py, px] = decay_val

        return torch.tensor(img_np, dtype=torch.float32)

    def __getitem__(self, idx):
        row = self.submission_df.iloc[idx]
        raw_path = row['path']
        path = raw_path[2:] if raw_path.startswith("./") else raw_path

        df = pd.read_csv(path)
        episodes = self.preprocessor.transform(df, is_train=False)

        if len(episodes) == 0:
            input_dim = self.preprocessor.get_input_dim()
            cont = torch.zeros((1, input_dim), dtype=torch.float32)
            cat = torch.zeros((1, 2), dtype=torch.long)
            img = torch.zeros((2, self.H, self.W), dtype=torch.float32)
        else:
            data = episodes[0]
            cont_np = data['cont']
            img = self._generate_image(cont_np)
            cont = torch.tensor(cont_np, dtype=torch.float32)
            cat = torch.tensor(data['cat'], dtype=torch.long)

        return img, cont, cat


def test_mm_collate_fn(batch):
    imgs, conts, cats = zip(*batch)
    imgs_batched = torch.stack(imgs, dim=0)
    lengths = torch.tensor([len(c) for c in conts], dtype=torch.long)
    conts_padded = pad_sequence(conts, batch_first=True)
    cats_padded = pad_sequence(cats, batch_first=True)
    return imgs_batched, conts_padded, cats_padded, lengths


# =========================
# Load Submission Meta
# =========================
submission = pd.read_csv("sample_submission.csv")
test_meta = pd.read_csv("test.csv")
submission = submission.merge(test_meta, on="game_episode", how="left")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# 10-Fold Inference with CORRECTED TTA
# =========================
NUM_FOLDS = 10
all_fold_preds = []

for fold in range(1, NUM_FOLDS + 1):
    print(f"\n=== Inference Fold {fold} with Corrected TTA ===")

    # 1. Load preprocessor
    preprocessor = joblib.load(f"models/preprocessor_v7_h96_{fold}.pkl")

    # 2. Dataset / Loader
    test_dataset = TestMultiModalDataset(submission, preprocessor)
    test_loader = DataLoader(
        test_dataset,
        batch_size=128,
        shuffle=False,
        collate_fn=test_mm_collate_fn
    )

    # 3. Load model
    input_dim_cont = preprocessor.get_input_dim()
    num_types, num_results = preprocessor.get_num_classes()

    model = MultiModalNetV6(input_dim_cont, num_types, num_results).to(DEVICE)
    model.load_state_dict(torch.load(f"models/multimodal_v7_h96_{fold}.pth", map_location=DEVICE))
    model.eval()

    fold_preds = []

    # 4. Inference with CORRECTED TTA
    with torch.no_grad():
        for imgs, cont, cat, lengths in tqdm(test_loader, desc=f"Fold {fold}"):
            imgs, cont, cat, lengths = (
                imgs.to(DEVICE),
                cont.to(DEVICE),
                cat.to(DEVICE),
                lengths.to(DEVICE),
            )

            # -------------------------
            # (1) 원본 예측
            # -------------------------
            pred1, _ = model(imgs, cont, cat, lengths)

            # -------------------------
            # (2) Y-Flip TTA 예측 (CORRECTED!)
            # -------------------------
            imgs_f = torch.flip(imgs, dims=[2])  # Y-axis flip

            # ✅ CORRECTED: Raw → Flip → Scale
            scaler = preprocessor.scaler
            mean = torch.tensor(scaler.mean_, device=cont.device, dtype=cont.dtype)
            std = torch.tensor(scaler.scale_, device=cont.device, dtype=cont.dtype)

            # Step 1: z-score → raw(norm)
            cont_raw = cont * std + mean

            # Step 2: Y-Flip in raw space
            cont_raw[:, :, 1] = 1.0 - cont_raw[:, :, 1]  # start_y_norm
            cont_raw[:, :, 3] = 1.0 - cont_raw[:, :, 3]  # end_y_prev_norm
            cont_raw[:, :, 5] = -cont_raw[:, :, 5]       # dy_prev_norm

            # Step 3: raw → z-score
            cont_f = (cont_raw - mean) / std

            pred2, _ = model(imgs_f, cont_f, cat, lengths)

            # 예측값 다시 flip
            pred2[:, 1] = 1.0 - pred2[:, 1]

            # -------------------------
            # (3) 평균
            # -------------------------
            pred = 0.5 * (pred1 + pred2)

            pred_np = pred.cpu().numpy()
            pred_np[:, 0] *= 105.0
            pred_np[:, 1] *= 68.0
            fold_preds.append(pred_np)

    fold_preds = np.vstack(fold_preds)
    all_fold_preds.append(fold_preds)

# =========================
# Ensemble (Mean)
# =========================
final_preds = np.mean(all_fold_preds, axis=0)

submission["end_x"] = final_preds[:, 0].clip(0, 105)
submission["end_y"] = final_preds[:, 1].clip(0, 68)

submission[["game_episode", "end_x", "end_y"]].to_csv(
    "submission_multimodal/multimodal_v7_h96_tta_corrected.csv", index=False
)

print("\n✅ Saved: submission_multimodal/multimodal_v7_h96_tta_corrected.csv")
print("📊 TTA Fix Applied: Raw → Flip → Scale")

C:\Users\semic\AppData\Local\Temp\ipykernel_83332\246526493.py:116: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"models/multimodal_v7_h96


=== Inference Fold 1 with Corrected TTA ===


Fold 1: 100%|██████████| 19/19 [01:13<00:00,  3.89s/it]
C:\Users\semic\AppData\Local\Temp\ipykernel_83332\246526493.py:116: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mod


=== Inference Fold 2 with Corrected TTA ===


Fold 2: 100%|██████████| 19/19 [00:45<00:00,  2.39s/it]
C:\Users\semic\AppData\Local\Temp\ipykernel_83332\246526493.py:116: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mod


=== Inference Fold 3 with Corrected TTA ===


Fold 3:   5%|▌         | 1/19 [00:02<00:42,  2.36s/it]

In [10]:
result = pd.read_csv("submission_multimodal.csv")
result.head()

,game_episode,end_x,end_y
0,153363_1,63.607921,11.172691
1,153363_2,24.612310,56.282756
2,153363_6,31.860115,64.409105
3,153363_7,52.938785,5.637901
4,153363_8,82.286735,7.873227


In [11]:
result_100 = pd.read_csv("submission_multimodal/submission_multimodal_epochs100.csv")
result_100.head()

,game_episode,end_x,end_y
0,153363_1,62.625419,8.007021
1,153363_2,21.414417,53.239441
2,153363_6,32.698217,64.080480
3,153363_7,57.145877,3.871556
4,153363_8,82.507950,8.292343


In [12]:
result_100_32 = pd.read_csv("submission_multimodal/submission_multimodal_epochs100_32.csv")
result_100_32.head()

,game_episode,end_x,end_y
0,153363_1,66.829318,8.926390
1,153363_2,25.648449,54.653327
2,153363_6,33.047745,64.115743
3,153363_7,58.226136,4.716273
4,153363_8,82.273184,7.475297


In [13]:
result_100_32_v6 = pd.read_csv("submission_multimodal/submission_multimodal_epochs100_32_v6.csv")
result_100_32_v6.head()

,game_episode,end_x,end_y
0,153363_1,64.597333,8.189129
1,153363_2,26.559090,50.912119
2,153363_6,31.622801,61.919090
3,153363_7,52.369850,3.774227
4,153363_8,84.277331,8.135580


In [14]:
result_v7 = pd.read_csv("submission_multimodal/multimodal_v7.csv")
result_v7.head()

,game_episode,end_x,end_y
0,153363_1,64.898580,9.964928
1,153363_2,26.757336,52.302315
2,153363_6,29.749920,63.453846
3,153363_7,56.788654,5.046815
4,153363_8,82.100350,8.899534


In [15]:
result_v7_fold = pd.read_csv("submission_multimodal/multimodal_v7_fold10.csv")
result_v7_fold.head()

,game_episode,end_x,end_y
0,153363_1,62.648010,9.487803
1,153363_2,26.349102,50.654480
2,153363_6,30.616455,63.448414
3,153363_7,53.435190,4.815488
4,153363_8,81.505226,8.122751


In [16]:
result_v7_fold_tta = pd.read_csv("submission_multimodal/multimodal_v7_fold10_tta.csv")
result_v7_fold_tta.head()

,game_episode,end_x,end_y
0,153363_1,62.602947,10.580890
1,153363_2,26.541010,49.843010
2,153363_6,30.677760,63.525776
3,153363_7,54.284637,4.835038
4,153363_8,81.609910,8.414079


In [17]:
result_v7_foldseed = pd.read_csv("submission_multimodal/multimodal_v7_foldseed.csv")
result_v7_foldseed.head()

,game_episode,end_x,end_y
0,153363_1,63.421947,10.237642
1,153363_2,26.759787,49.413464
2,153363_6,29.883270,62.548830
3,153363_7,55.195984,5.207705
4,153363_8,82.545845,8.009755


In [18]:
result_v7_foldseed_tta = pd.read_csv("submission_multimodal/multimodal_v7_foldseed_tta.csv")
result_v7_foldseed_tta.head()

,game_episode,end_x,end_y
0,153363_1,63.095875,11.823668
1,153363_2,26.836575,49.308430
2,153363_6,30.428274,63.259560
3,153363_7,56.232758,5.196191
4,153363_8,82.745255,8.249907


In [19]:
result_v7_h96= pd.read_csv("submission_multimodal/multimodal_v7_h96.csv")
result_v7_h96.head()

,game_episode,end_x,end_y
0,153363_1,64.615486,10.832696
1,153363_2,25.001268,49.346035
2,153363_6,27.185059,63.219980
3,153363_7,55.024788,5.205964
4,153363_8,81.562614,8.390555


In [20]:
result_v7_foldseed_h96_tta = pd.read_csv("submission_multimodal/multimodal_v7_h96_tta.csv")
result_v7_foldseed_h96_tta.head()

,game_episode,end_x,end_y
0,153363_1,64.885360,11.527712
1,153363_2,25.639957,48.075230
2,153363_6,28.853336,63.301735
3,153363_7,53.528250,5.063232
4,153363_8,81.424120,8.359509


In [ ]:
result_v7_foldseed_h96_tta_corrected = pd.read_csv("submission_multimodal/multimodal_v7_h96_tta_corrected.csv")
result_v7_foldseed_h96_tta_corrected.head()

,game_episode,end_x,end_y
0,153363_1,64.885360,11.527712
1,153363_2,25.639957,48.075230
2,153363_6,28.853336,63.301735
3,153363_7,53.528250,5.063232
4,153363_8,81.424120,8.359509
